In [ ]:
!pip install -q diffusers transformers accelerate imageio[ffmpeg] torch safetensors
import torch
from diffusers import MotionAdapter, AnimateDiffPipeline, DDIMScheduler
from diffusers.utils import export_to_video
from google.colab import files


In [ ]:
print("--> Loading Motion Adapter...")
adapter = MotionAdapter.from_pretrained("guoyww/animatediff-motion-adapter-v1-5-2", torch_dtype=torch.float16)

print("--> Loading Anime Base Model...")
model_id = "stablediffusionapi/anything-v5"
pipe = AnimateDiffPipeline.from_pretrained(
    model_id,
    motion_adapter=adapter,
    torch_dtype=torch.float16
)
pipe.scheduler = DDIMScheduler.from_pretrained(model_id, subfolder="scheduler", clip_sample=False, timestep_spacing="linspace", beta_schedule="linear", steps_offset=1)

pipe.enable_vae_slicing()
pipe.enable_model_cpu_offload()

prompt = "masterpiece, best quality, 2D hand-drawn Japanese fighting animation style, a stylish confident young man with dark brown hair wearing a black oversized sweatshirt and black bandana covering lower face, looking straight at camera. Suddenly a bright bubbly girl with long light brown blue hair in a green long-sleeve top leaps out playfully from behind him. Radiant warm neon glow background with dynamic spark lines. Clean line art, pastel streetwear aesthetic"
negative_prompt = "3d, cgi, photorealistic, worst quality, low quality, bad anatomy, blurry, distorted, artifacts, extra limbs, bad proportions"

print("--> Generating high-fidelity anime video...")
result = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_frames=16,
    guidance_scale=7.5,
    num_inference_steps=25
)

output_filename = "high_quality_anime.mp4"
export_to_video(result.frames[0], output_filename, fps=8)
print(f"--> Success! Exported to {output_filename}")
files.download(output_filename)